# Exercise 3: Semantic similarity

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Meaning and Retrieval · version 0.1.0*

> Compare encoders on the Kardeş-NLU STS test sets with Spearman's ρ and bootstrap intervals. Then design a 50-pair STS set for a Turkic language without one: two native raters, a 0–5 scale, and quadratic-weighted κ for their agreement.

**Setup.** `pip install torch transformers sentencepiece scipy bm25s requests "turkicnlp[hfst]==0.5.0"` (the compiled `hfst` package needs Python 3.11 or 3.12). Encoders are loaded at the Hugging Face revisions pinned in `MODELS`, from a local Hugging Face cache (`HF_HUB_DIR`), a folder of copies (`MODELS_DIR`) or the Hub; NLLB-200 is read from TurkicNLP's model folder (`~/.turkicnlp/models/huggingface/facebook--nllb-200-distilled-600M`) as the `embeddings` processor does. Data come from commit-pinned GitHub repositories: SIB-200 (FLORES-200 sentences, the same 1,004 sentences in every language), Kardeş-NLU's STS sets and KazQAD's retrieval collection (825,309 Kazakh Wikipedia passages, about 230 MB compressed). Embeddings, lemma maps and retrieval runs are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`). Everything runs on the CPU.

Each Kardeş-NLU STS test set has 800 sentence pairs translated from English STS data, with the English gold similarity scores (0–5). Cosine similarity between the two sentence vectors is correlated with the gold score; the English originals give a reference.

In [ ]:
import collections, csv, gzip, hashlib, io, json, math, os, random, re, statistics, time
from pathlib import Path
import numpy as np, requests

SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
KARDES = "https://raw.githubusercontent.com/lksenel/Kardes-NLU/48eea77fda904e731d228de833f4c57d130fad9c/Data/"
KAZQAD = "https://raw.githubusercontent.com/IS2AI/KazQAD/ab8684a667109f2a7345f4f89ab3b123e8d78e1f/data/information-retrieval/"
MODELS = {   # sentence encoder -> pinned revision (mean pooling, 128-token ceiling, as configured)
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": "e8f8c211226b894fcb81acc59f3b34ba3efd5f42",
    "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "c2e19453aae46f98a865abfdf173f35f2e119f5a",
    "cis-lmu/glot500-base": "d4d7c1ec01828fdf7452a4ccf7b55177aced175e",                 # tokenizer only
    "meta-llama/Meta-Llama-3-8B": "8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920",          # tokenizer only
    "allenai/Olmo-3-1025-7B": "0a15c289b17ad4f27aabc863276ec525d5312c01",              # tokenizer only
    "Qwen/Qwen3.6-27B": "6a9e13bd6fc8f0983b9b99948120bc37f49c13e9",                    # tokenizer only
}
NAMES = {"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": "MiniLM",
         "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "XLM-R para", "nllb": "NLLB enc"}
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
HF_HUB_DIR = Path(os.environ.get("HF_HUB_DIR", "~/.cache/huggingface/hub")).expanduser()
MODELS_DIR = Path(os.environ.get("MODELS_DIR", CACHE / "models"))
NLLB_DIR = Path("~/.turkicnlp/models/huggingface/facebook--nllb-200-distilled-600M").expanduser()
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=600); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    FETCHED[url] = h.hexdigest()
    return path

def model_source(name):
    rev = MODELS[name]
    for p in (HF_HUB_DIR / f"models--{name.replace('/', '--')}" / "snapshots" / rev,
              MODELS_DIR / f"models--{name.replace('/', '--')}--{rev}"):
        if (p / "config.json").exists():
            return str(p), None
    return name, rev

def tokenizer(name):
    from transformers import AutoTokenizer
    path, rev = model_source(name)
    return AutoTokenizer.from_pretrained(path, revision=rev)

def sib(lang):
    """All 1,004 SIB-200 sentences of a FLORES language (train, dev, test), sorted by index_id."""
    rows = []
    for split in ("train", "dev", "test"):
        text = fetch(f"{SIB}{lang}/{split}.tsv").read_text(encoding="utf-8")
        rows += [(int(r["index_id"]), r["text"]) for r in csv.DictReader(io.StringIO(text), delimiter="\t")]
    return [t for _, t in sorted(rows)]

TRANSLIT = {"kaz_Cyrl": ("kaz", "Cyrl"), "kir_Cyrl": ("kir", "Cyrl"), "tat_Cyrl": ("tat", "Cyrl"),
            "bak_Cyrl": ("bak", "Cyrl"), "uzn_Latn": ("uzb", "Latn"), "azj_Latn": ("aze", "Latn"), "tur_Latn": ("tur", "Latn")}
_T = {}
def to_cts(lang, text):
    """Common Turkic Alphabet version of a sentence with TurkicNLP's transliterator."""
    if lang not in _T:
        from turkicnlp.scripts import Script
        from turkicnlp.scripts.transliterator import Transliterator
        code, scr = TRANSLIT[lang]
        _T[lang] = Transliterator(code, Script.CYRILLIC if scr == "Cyrl" else Script.LATIN, Script.COMMON_TURKIC)
    return _T[lang].transliterate(text)

_ENC = {}
NLLB_CODE = {"kaz_Cyrl": "kaz_Cyrl", "kir_Cyrl": "kir_Cyrl", "tat_Cyrl": "tat_Cyrl", "bak_Cyrl": "bak_Cyrl",
             "uzn_Latn": "uzn_Latn", "azj_Latn": "azj_Latn", "tur_Latn": "tur_Latn", "rus_Cyrl": "rus_Cyrl",
             "eng_Latn": "eng_Latn", "uig_Arab": "uig_Arab"}
def embed(model, texts, lang=None, batch=32):
    """L2-normalised mean-pooled sentence vectors. Paraphrase encoders truncate at their configured
    128 tokens; the NLLB encoder (as in TurkicNLP's embeddings processor) gets the language token of
    `lang` and is not truncated below 512. Cached on disk by model, language and text hash."""
    key = hashlib.sha1(json.dumps([model, lang, texts], ensure_ascii=False).encode()).hexdigest()[:20]
    path = CACHE / "ch13-vectors" / f"{key}.npy"
    if path.exists():
        return np.load(path)
    import torch
    torch.set_num_threads(THREADS)
    if model not in _ENC:
        from transformers import AutoModel, AutoTokenizer
        if model == "nllb":
            from transformers import AutoModelForSeq2SeqLM
            tok = AutoTokenizer.from_pretrained(NLLB_DIR)
            enc = AutoModelForSeq2SeqLM.from_pretrained(NLLB_DIR).get_encoder().eval()
            _ENC[model] = (tok, enc, 512)
        else:
            src, rev = model_source(model)
            _ENC[model] = (AutoTokenizer.from_pretrained(src, revision=rev), AutoModel.from_pretrained(src, revision=rev).eval(), 128)
    tok, enc, limit = _ENC[model]
    if model == "nllb":
        tok.src_lang = NLLB_CODE[lang]
    out = []
    with torch.inference_mode():
        for i in range(0, len(texts), batch):
            b = tok(texts[i:i + batch], padding=True, truncation=True, max_length=limit, return_tensors="pt")
            h = enc(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).to(h.dtype)
            v = (h * m).sum(1) / m.sum(1).clamp(min=1)
            out.append(torch.nn.functional.normalize(v, dim=-1).numpy())
    vec = np.concatenate(out).astype(np.float32)
    path.parent.mkdir(parents=True, exist_ok=True); np.save(path, vec)
    return vec

def top1(q, c, targets):
    """Share of queries whose nearest candidate is the aligned one (exact duplicate target texts count)."""
    best = (q @ c.T).argmax(1)
    return float(np.mean([targets[b] == targets[i] for i, b in enumerate(best)]))

print("encoders:", ", ".join(f"{NAMES.get(n, n)} ({'local' if model_source(n)[1] is None else 'Hub'})" for n in list(MODELS)[:2]),
      f"+ NLLB encoder ({'local' if (NLLB_DIR / 'config.json').exists() else 'missing'})")

In [ ]:
from scipy.stats import spearmanr
LANG = {"kazakh": ("kk", "kaz_Cyrl"), "kyrgyz": ("ky", "kir_Cyrl"), "azeri": ("az", "azj_Latn"),
        "uzbek": ("uz", "uzn_Latn"), "uyghur": ("ug", "uig_Arab")}
def sts(folder, code):
    text = fetch(f"{KARDES}{folder}/sts.test.{code}.csv").read_text(encoding="utf-8-sig")
    return list(csv.DictReader(io.StringIO(text)))
ENCODERS = ["sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", "sentence-transformers/paraphrase-xlm-r-multilingual-v1", "nllb"]
rng = np.random.default_rng(0)
def rho_ci(sim, gold, n=1000):
    idx = rng.integers(0, len(gold), size=(n, len(gold)))
    boots = [spearmanr(sim[i], gold[i]).statistic for i in idx]
    return spearmanr(sim, gold).statistic, np.percentile(boots, [2.5, 97.5])
print("Kardeş-NLU STS test sets: Spearman's rho x 100 between cosine similarity and the gold score, bootstrap 95% CI")
print(f"{'language':<9}{'pairs':>6}" + "".join(f"{NAMES[m]:>20}" for m in ENCODERS))
english = None
for folder, (code, flores) in LANG.items():
    rows = sts(folder, code)
    gold = np.array([float(r["score"]) for r in rows])
    cells = []
    for m in ENCODERS:
        a = embed(m, [r["s1_translation"] for r in rows], flores); b = embed(m, [r["s2_translation"] for r in rows], flores)
        r, (lo, hi) = rho_ci((a * b).sum(1), gold)
        cells.append(f"{100 * r:6.1f} [{100 * lo:4.1f}-{100 * hi:4.1f}]")
    print(f"{folder:<9}{len(rows):>6}" + "".join(f"{c:>20}" for c in cells))
    english = english or rows
gold = np.array([float(r["score"]) for r in english])
cells = []
for m in ENCODERS:
    a = embed(m, [r["sentence1"] for r in english], "eng_Latn"); b = embed(m, [r["sentence2"] for r in english], "eng_Latn")
    r, (lo, hi) = rho_ci((a * b).sum(1), gold)
    cells.append(f"{100 * r:6.1f} [{100 * lo:4.1f}-{100 * hi:4.1f}]")
print(f"{'English':<9}{len(english):>6}" + "".join(f"{c:>20}" for c in cells) + "   (originals of the Kazakh set)")
base = [(r["sentence1"], r["sentence2"], r["score"]) for r in sts("kazakh", "kk")]
same = {f: sum(x == (r["sentence1"], r["sentence2"], r["score"]) for x, r in zip(base, sts(f, c))) for f, (c, _) in LANG.items()}
print("rows whose English pair and gold score equal the Kazakh set's, in order:", same)

def weighted_kappa(a, b, k=6):
    """Quadratic-weighted Cohen's kappa for two raters' integer scores 0..k-1 (for a new STS set)."""
    a, b = np.asarray(a), np.asarray(b); o = np.zeros((k, k))
    for x, y in zip(a, b): o[x, y] += 1
    e = np.outer(np.bincount(a, minlength=k), np.bincount(b, minlength=k)) / len(a)
    w = np.subtract.outer(np.arange(k), np.arange(k)) ** 2
    return 1 - (w * o).sum() / (w * e).sum()
print("weighted kappa check: identical ratings ->", weighted_kappa([0, 1, 2, 3, 4, 5], [0, 1, 2, 3, 4, 5]),
      "| shifted by one ->", round(weighted_kappa([0, 1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 5]), 3))

## Worked solution

Output of `experiments/ch13_exercises/run_ch13_exercises.py`:

```
Kardeş-NLU STS test sets: Spearman's rho x 100 between cosine similarity and the gold score, bootstrap 95% CI
language  pairs              MiniLM          XLM-R para            NLLB enc
kazakh      800    61.2 [56.0-65.4]    73.5 [69.5-77.0]    74.8 [70.8-78.2]
kyrgyz      800    50.1 [44.9-55.4]    62.4 [57.5-66.9]    69.2 [65.0-72.9]
azeri       800    58.1 [53.0-62.5]    70.6 [66.4-74.3]    66.1 [61.3-70.3]
uzbek       800    36.9 [30.6-42.9]    51.7 [46.0-57.4]    66.3 [61.8-70.8]
uyghur      800    37.8 [32.0-43.8]    39.3 [33.4-45.3]    58.4 [53.2-62.8]
English     800    83.9 [81.4-86.2]    83.0 [80.3-85.4]    71.8 [67.7-75.4]   (originals of the Kazakh set)
rows whose English pair and gold score equal the Kazakh set's, in order: {'kazakh': 800, 'kyrgyz': 798, 'azeri': 800, 'uzbek': 800, 'uyghur': 796}
weighted kappa check: identical ratings -> 1.0 | shifted by one -> 0.857
```

**Point estimates.** NLLB has the highest reported Spearman correlation among these three systems for Kazakh, Kyrgyz, Uzbek and Uyghur; XLM-R leads for Azerbaijani. The two paraphrase encoders lead on the English originals. This ordering does not identify why models differ or establish that every difference is reliable.

**Uncertainty.** The reported bootstrap intervals describe individual correlations. To assess a model difference, bootstrap the paired difference on the same sentence pairs. Overlapping or non-overlapping marginal intervals are not a substitute for that comparison.

**Translated labels.** The releases retain English STS scores. Translation can change similarity, so inspect translated pairs before treating those scores as target-language judgements. Between 796 and 800 rows match the Kazakh set’s English pairs and scores in order; align explicitly for cross-language comparisons.

**A native pilot.** Write 50 pairs spanning paraphrase, partial overlap, shared topic and unrelated meaning. Have two fluent raters score them independently after calibrating on separate examples. Report rating distributions, disagreements and quadratic-weighted kappa; there is no universal kappa threshold that certifies gold data. Fifty pairs form a pilot, with uncertainty depending on the observed scores and sampling.

## Your turn — extension

1. Compare cosine similarity on the Kazakh pairs with their Cyrillic-to-CTS conversions.
2. Write ten Tatar sentence pairs, score them yourself and compute Spearman's ρ for one encoder.